# Day 08: 对齐技术全景图 — 从 RLHF 到 GRPO

**学习目标：**
- 了解对齐技术的演进历程和各方法之间的关系
- 掌握 RLHF / DPO / KTO / ORPO / SimPO / CAI / GRPO 的核心思想
- 学会根据场景选择合适的对齐方法

---

## 1. 对齐技术全景图

### 1.1 技术演进时间线

| 年份 | 方法 | 关键改进 | 代表工作 |
|------|------|---------|----------|
| 2022 | **RLHF** | 首次大规模应用人类偏好训练 | InstructGPT, ChatGPT |
| 2023 | **DPO** | 去掉 RM 和 PPO，直接优化 | Rafailov et al. |
| 2023 | **CAI** | 用 AI 替代人类标注 | Anthropic (Constitutional AI) |
| 2024 | **KTO** | 不需要配对数据 | Ethayarajh et al. |
| 2024 | **ORPO** | SFT + 对齐一步完成 | Hong et al. |
| 2024 | **SimPO** | 去掉参考模型 + 长度归一化 | Meng et al. |
| 2024 | **GRPO** | 组内排名奖励，适合推理任务 | DeepSeek-R1 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 绘制对齐技术演进图
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(-1, 11)
ax.set_ylim(-1, 9)
ax.axis("off")
ax.set_title("LLM 对齐技术演进全景图", fontsize=18, fontweight="bold", pad=20)

# 节点定义
nodes = {
    "RLHF":  (1, 7, "RLHF\n(2022)", "#FF6B6B"),
    "DPO":   (4, 7, "DPO\n(2023)", "#4ECDC4"),
    "CAI":   (1, 4, "CAI\n(2023)", "#DDA0DD"),
    "KTO":   (3, 4, "KTO\n(2024)", "#45B7D1"),
    "ORPO":  (5, 4, "ORPO\n(2024)", "#96CEB4"),
    "SimPO": (7, 4, "SimPO\n(2024)", "#FFEAA7"),
    "GRPO":  (7, 7, "GRPO\n(2024)", "#FF8C42"),
}

descs = {
    "RLHF": "SFT->RM->PPO\n三阶段流程",
    "DPO": "直接偏好优化\n无需RM和PPO",
    "CAI": "AI 反馈\n替代人类标注",
    "KTO": "单条标注\n前景理论",
    "ORPO": "SFT+对齐\n一步完成",
    "SimPO": "长度归一化\n无需Ref模型",
    "GRPO": "组内排名\nDeepSeek-R1",
}

for key, (x, y, label, color) in nodes.items():
    circle = plt.Circle((x, y), 0.8, color=color, alpha=0.85, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y + 0.1, label, ha="center", va="center", fontsize=9, fontweight="bold", zorder=4)
    ax.text(x, y - 1.3, descs[key], ha="center", va="top", fontsize=7, style="italic", color="#555",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

# 演进箭头
edges = [
    ("RLHF", "DPO", "去除RM+PPO"),
    ("RLHF", "CAI", "AI替代人类"),
    ("DPO", "KTO", "去除配对"),
    ("DPO", "ORPO", "去除Ref"),
    ("DPO", "SimPO", "长度归一化"),
    ("RLHF", "GRPO", "去除Critic"),
]

for src, dst, label in edges:
    sx, sy = nodes[src][0], nodes[src][1]
    dx, dy = nodes[dst][0], nodes[dst][1]
    ax.annotate("", xy=(dx, dy + 0.8 if dy < sy else dy - 0.8),
                xytext=(sx, sy - 0.8 if dy < sy else sy + 0.8),
                arrowprops=dict(arrowstyle="->", color="#666", lw=1.5, connectionstyle="arc3,rad=0.1"), zorder=2)
    mx, my = (sx + dx) / 2, (sy + dy) / 2
    ax.text(mx, my, label, ha="center", va="center", fontsize=7, color="#333",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8, edgecolor="#ccc"))

# 分类区域
rect1 = mpatches.FancyBboxPatch((-0.3, 5.8), 8.5, 2.8, boxstyle="round,pad=0.3",
                                 facecolor="#FFE0E0", alpha=0.2, edgecolor="#FF6B6B", ls="--", zorder=1)
ax.add_patch(rect1)
ax.text(-0.1, 8.4, "RL-based", fontsize=10, color="#FF6B6B", fontweight="bold")

rect2 = mpatches.FancyBboxPatch((1.5, 2.5), 7, 2.3, boxstyle="round,pad=0.3",
                                 facecolor="#E0FFE0", alpha=0.2, edgecolor="#4ECDC4", ls="--", zorder=1)
ax.add_patch(rect2)
ax.text(1.7, 4.6, "Preference-based (DPO 变体)", fontsize=10, color="#4ECDC4", fontweight="bold")

# 时间轴
ax.annotate("", xy=(10, 0.5), xytext=(0, 0.5),
            arrowprops=dict(arrowstyle="->", color="gray", lw=2))
for year, xp in [(2022, 1), (2023, 3.5), (2024, 7)]:
    ax.text(xp, 0.6, str(year), ha="center", fontsize=9, color="gray", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 2. 每种方法详解

### 2.1 RLHF (Reinforcement Learning from Human Feedback)

**核心思想：** 用人类偏好训练奖励模型，再用 RL (PPO) 优化策略。

**流程：** SFT → 训练 RM → PPO 优化

**损失函数：**
$$\mathcal{L}_{PPO} = \mathbb{E}\left[\min(r_t \hat{A}_t,\ \text{clip}(r_t, 1\pm\epsilon)\hat{A}_t)\right] - \beta \cdot KL(\pi_\theta \| \pi_{ref})$$

In [ ]:
# RLHF 伪代码
print("""
RLHF 训练伪代码：
══════════════════════════════════════════════════════

# Stage 1: SFT
sft_model = pretrained_model.finetune(sft_data)

# Stage 2: Reward Model
reward_model = sft_model.add_reward_head()  # 加一个标量输出头
for (prompt, y_w, y_l) in preference_data:
    r_w = reward_model(prompt, y_w)  # 好回复的奖励
    r_l = reward_model(prompt, y_l)  # 差回复的奖励
    loss = -log(sigmoid(r_w - r_l))  # Bradley-Terry 损失
    loss.backward()

# Stage 3: PPO
policy = sft_model.copy()       # 当前策略 π_θ
ref_model = sft_model.freeze()  # 参考策略 π_ref (冻结)
for prompt in prompts:
    response = policy.generate(prompt)       # 采样回复
    reward = reward_model(prompt, response)  # 奖励打分
    kl = KL(policy, ref_model)               # KL 散度
    objective = reward - beta * kl            # PPO 目标
    ppo_update(policy, objective)             # PPO 更新
""")

### 2.2 DPO (Direct Preference Optimization)

**核心思想：** 跳过 RM 和 PPO，直接用偏好数据优化策略。

**关键洞见：** 最优策略和最优奖励函数一一对应，可以把奖励模型「折叠」进策略优化中。

**损失函数：**
$$\mathcal{L}_{DPO} = -\mathbb{E}\left[\log \sigma\left(\beta \left(\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right)\right]$$

In [ ]:
# DPO 伪代码
print("""
DPO 训练伪代码：
══════════════════════════════════════════════════════

# Step 1: SFT
sft_model = pretrained_model.finetune(sft_data)

# Step 2: DPO 直接优化
policy = sft_model.copy()       # π_θ
ref_model = sft_model.freeze()  # π_ref (冻结)

for (prompt, y_w, y_l) in preference_data:
    # 计算对数概率
    logp_w = policy.log_prob(y_w | prompt)      # π_θ(y_w|x)
    logp_l = policy.log_prob(y_l | prompt)      # π_θ(y_l|x)
    ref_logp_w = ref_model.log_prob(y_w | prompt)  # π_ref(y_w|x)
    ref_logp_l = ref_model.log_prob(y_l | prompt)  # π_ref(y_l|x)
    
    # 计算隐式奖励差
    logits = beta * ((logp_w - ref_logp_w) - (logp_l - ref_logp_l))
    
    # DPO 损失
    loss = -log_sigmoid(logits)
    loss.backward()
""")

### 2.3 KTO (Kahneman-Tversky Optimization)

**核心思想：** 不需要配对的偏好数据，只需要「好/坏」标签。基于行为经济学的前景理论。

**关键改进：** DPO 要求每条数据都有 (chosen, rejected) 配对，但实际中获取配对数据成本高。KTO 只需要 (response, good/bad)。

**损失函数：**
$$\mathcal{L}_{KTO} = \mathbb{E}_{y_w}\left[1-\sigma(\beta \cdot r_w)\right] + \mathbb{E}_{y_l}\left[1-\sigma(-\beta \cdot r_l)\right]$$

In [ ]:
# KTO 伪代码
print("""
KTO 训练伪代码：
══════════════════════════════════════════════════════

policy = sft_model.copy()
ref_model = sft_model.freeze()

for (prompt, response, label) in labeled_data:
    # 计算隐式奖励
    r = beta * (policy.log_prob(response|prompt) - ref.log_prob(response|prompt))
    r = r - KL_baseline  # 减去 KL 基线
    
    if label == "good":
        loss = 1 - sigmoid(r)      # 好样本：希望 r 高
    else:  # label == "bad"
        loss = 1 - sigmoid(-r)     # 坏样本：希望 r 低
    
    loss.backward()

# 优势：不需要 (chosen, rejected) 配对！
#        只需要 (response, good/bad) 标签
""")

### 2.4 ORPO (Odds Ratio Preference Optimization)

**核心思想：** 把 SFT 和偏好优化合并为一步，不需要参考模型。

**关键改进：** DPO 需要先 SFT 再 DPO（两步），且需要参考模型。ORPO 用赔率比（odds ratio）作为偏好信号，一步完成。

**损失函数：**
$$\mathcal{L}_{ORPO} = \mathcal{L}_{SFT} + \lambda \cdot \mathcal{L}_{OR}$$
$$\mathcal{L}_{OR} = -\log\sigma\left(\log\frac{\text{odds}_\theta(y_w|x)}{\text{odds}_\theta(y_l|x)}\right)$$

In [ ]:
# ORPO 伪代码
print("""
ORPO 训练伪代码：
══════════════════════════════════════════════════════

model = pretrained_model  # 不需要先 SFT！

for (prompt, y_w, y_l) in preference_data:
    # SFT 损失（在 chosen 上）
    sft_loss = -log_prob(y_w | prompt)
    
    # 计算赔率 odds = p / (1-p)
    p_w = exp(log_prob(y_w | prompt))
    p_l = exp(log_prob(y_l | prompt))
    odds_w = p_w / (1 - p_w)
    odds_l = p_l / (1 - p_l)
    
    # 赔率比损失
    or_loss = -log_sigmoid(log(odds_w / odds_l))
    
    # 总损失 = SFT + λ * 赔率比
    loss = sft_loss + lambda_ * or_loss
    loss.backward()

# 优势：不需要参考模型！不需要先做 SFT！
""")

### 2.5 SimPO (Simple Preference Optimization)

**核心思想：** 去掉参考模型 + 用长度归一化消除长度偏差。

**关键改进：** DPO 有长度偏差（长回复的对数概率天然更低），SimPO 用序列长度做归一化。

**损失函数：**
$$\mathcal{L}_{SimPO} = -\mathbb{E}\left[\log\sigma\left(\frac{\beta}{|y_w|}\log\pi_\theta(y_w|x) - \frac{\beta}{|y_l|}\log\pi_\theta(y_l|x) - \gamma\right)\right]$$

In [ ]:
# SimPO 伪代码
print("""
SimPO 训练伪代码：
══════════════════════════════════════════════════════

model = sft_model  # 不需要参考模型！

for (prompt, y_w, y_l) in preference_data:
    # 计算长度归一化的对数概率
    logp_w = model.log_prob(y_w | prompt) / len(y_w)  # 除以长度！
    logp_l = model.log_prob(y_l | prompt) / len(y_l)
    
    # SimPO 损失（带 margin γ）
    loss = -log_sigmoid(beta * (logp_w - logp_l) - gamma)
    loss.backward()

# 优势：
#   1. 不需要参考模型（节省 50% 内存）
#   2. 长度归一化消除长度偏差
#   3. margin γ 提供更强的优化信号
""")

### 2.6 CAI (Constitutional AI)

**核心思想：** 用 AI 替代人类进行偏好标注（RLAIF），通过一组「宪法原则」指导 AI 标注。

**关键改进：** 人类标注成本高且不一致。CAI 用强大的 AI 模型根据预定义原则来判断回复好坏。

**流程：**
1. 定义宪法原则（如：回复应该有帮助、无害、诚实）
2. AI 根据原则对回复进行评判和修改
3. 用 AI 反馈训练模型（RLAIF）

In [ ]:
# CAI 伪代码
print("""
Constitutional AI 伪代码：
══════════════════════════════════════════════════════

# 定义宪法原则
constitution = [
    "回复应该有帮助且信息丰富",
    "回复不应包含有害或不道德的内容",
    "回复应该诚实，不确定时承认不知道",
    "回复应该尊重用户隐私",
]

# Step 1: Red-teaming（红队测试）
for prompt in harmful_prompts:
    response = model.generate(prompt)
    
    # AI 根据宪法评判
    critique = judge_model.evaluate(response, constitution)
    
    # AI 修改回复
    revised = judge_model.revise(response, critique)

# Step 2: RLAIF（用 AI 反馈替代人类反馈）
for (prompt, response_a, response_b) in pairs:
    # AI 打分
    preference = judge_model.compare(response_a, response_b, constitution)
    # 用 AI 偏好训练（和 RLHF/DPO 相同）
""")

### 2.7 GRPO (Group Relative Policy Optimization)

**核心思想：** 对每个 prompt 采样多个回复，用组内排名作为奖励信号，不需要 critic 模型。

**关键改进：** PPO 需要 critic 模型估计 value，GRPO 用同组内的相对排名代替。特别适合数学推理等有明确奖励函数的任务。

**损失函数：**
$$\mathcal{L}_{GRPO} = -\mathbb{E}\left[\sum_t \min(r_t \hat{A}_t,\ \text{clip}(r_t) \hat{A}_t)\right] + \beta \cdot KL$$
$$\hat{A}_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r})} \quad \text{(组内归一化)}$$

In [ ]:
# GRPO 伪代码
print("""
GRPO 训练伪代码 (DeepSeek-R1)：
══════════════════════════════════════════════════════

policy = sft_model.copy()
ref_model = sft_model.freeze()

for prompt in prompts:
    # 对每个 prompt 采样 G 个回复
    responses = [policy.generate(prompt) for _ in range(G)]
    
    # 用奖励函数打分（如：数学题正确性检查）
    rewards = [reward_fn(prompt, r) for r in responses]
    
    # 组内归一化（关键！）
    advantages = (rewards - mean(rewards)) / std(rewards)
    
    # PPO 风格更新（但不需要 critic！）
    for response, advantage in zip(responses, advantages):
        ratio = policy.prob(response) / old_policy.prob(response)
        clipped = clip(ratio, 1-eps, 1+eps)
        loss = -min(ratio * advantage, clipped * advantage)
        kl_penalty = beta * KL(policy, ref_model)
        total_loss = loss + kl_penalty
        total_loss.backward()

# 优势：
#   1. 不需要 critic 模型（节省内存和训练）
#   2. 组内归一化提供自然的基线
#   3. 适合有确定性奖励的任务（数学、编程）
""")

---
## 3. 如何选择对齐方法

### 3.1 对比表格

| 方法 | 需要 RM | 需要 Ref | 训练阶段 | 数据需求 | 内存 | 稳定性 | 适用场景 |
|------|---------|---------|---------|---------|------|--------|----------|
| **RLHF** | 是 | 是 | 3 | 偏好对+prompt | 高 | 低 | 大规模、资源充足 |
| **DPO** | 否 | 是 | 2 | 偏好对 | 中 | 高 | 通用首选 |
| **KTO** | 否 | 是 | 2 | 好/坏标签 | 中 | 高 | 无配对数据时 |
| **ORPO** | 否 | 否 | 1 | 偏好对 | 低 | 高 | 资源有限 |
| **SimPO** | 否 | 否 | 2 | 偏好对 | 低 | 高 | 对长度敏感时 |
| **CAI** | 可选 | 可选 | 2-3 | AI生成 | 中 | 中 | 无人工标注时 |
| **GRPO** | 否 | 是 | 2 | prompt+奖励函数 | 中 | 中 | 推理任务 |

In [ ]:
# 方法复杂度对比可视化
fig, ax = plt.subplots(figsize=(12, 6))

methods = ["RLHF", "DPO", "KTO", "ORPO", "SimPO", "CAI", "GRPO"]
complexity = [5, 3, 2, 2, 2, 4, 4]   # 实现复杂度
memory =     [5, 4, 3, 2, 2, 3, 3]   # 内存需求
data_req =   [4, 3, 2, 3, 3, 1, 2]   # 数据需求

x = np.arange(len(methods))
width = 0.22

ax.bar(x - width, complexity, width, label="实现复杂度", color="#FF6B6B", alpha=0.8)
ax.bar(x, memory, width, label="内存需求", color="#4ECDC4", alpha=0.8)
ax.bar(x + width, data_req, width, label="数据需求", color="#45B7D1", alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel("相对程度 (1-5)")
ax.set_title("对齐方法资源需求对比", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, 6)

plt.tight_layout()
plt.show()

### 3.2 决策树

```
你有什么数据？
│
├── 有配对偏好数据 (chosen, rejected)
│   ├── 内存充足？
│   │   ├── 是 → DPO （最稳定的通用选择）
│   │   └── 否 → SimPO 或 ORPO （不需要参考模型）
│   └── 还没做 SFT？
│       └── ORPO （SFT + 对齐一步完成）
│
├── 只有好/坏标签（无配对）
│   └── KTO
│
├── 有确定性奖励函数（数学/编程）
│   └── GRPO
│
├── 没有人工标注
│   └── CAI / RLAIF （用 AI 生成标注）
│
└── 大规模 + 不差钱
    └── RLHF （久经验证的方案）
```

### 3.3 工业界主流选择（2024-2025）

| 公司/项目 | 对齐方法 | 说明 |
|----------|---------|------|
| **OpenAI** (GPT-4) | RLHF | 最早的大规模实践者 |
| **Anthropic** (Claude) | CAI + RLHF | Constitutional AI + 传统 RLHF |
| **Meta** (Llama 3) | DPO | 从 RLHF 转向 DPO |
| **DeepSeek** (R1) | GRPO | 推理任务首选 |
| **HuggingFace** (Zephyr) | DPO | 开源社区标配 |
| **Mistral** | DPO | 简单高效 |

**趋势：** DPO 已成为中小规模模型的事实标准，GRPO 在推理任务中崛起。

In [ ]:
# 损失函数形状对比
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

x = np.linspace(-4, 4, 500)
beta = 0.1

# DPO vs IPO
ax = axes[0, 0]
dpo_loss = -np.log(1 / (1 + np.exp(-beta * x)) + 1e-10)
ipo_loss = (x - 1 / (2 * beta)) ** 2
ipo_loss = ipo_loss / ipo_loss.max() * dpo_loss.max()
ax.plot(x, dpo_loss, "b-", lw=2, label="DPO")
ax.plot(x, ipo_loss, "r--", lw=2, label="IPO")
ax.set_title("DPO vs IPO", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

# SimPO 长度归一化
ax = axes[0, 1]
for l in [10, 50, 100, 200]:
    simpo = -np.log(1 / (1 + np.exp(-(beta / l) * x * l)) + 1e-10)
    ax.plot(x, simpo, lw=1.5, label=f"len={l}")
ax.set_title("SimPO: 长度归一化效果", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

# KTO 好/坏独立损失
ax = axes[1, 0]
r = np.linspace(-3, 3, 500)
kto_good = 1 - 1 / (1 + np.exp(-beta * r))
kto_bad = 1 - 1 / (1 + np.exp(beta * r))
ax.plot(r, kto_good, "g-", lw=2, label="好样本 (希望 r 高)")
ax.plot(r, kto_bad, "r-", lw=2, label="坏样本 (希望 r 低)")
ax.set_title("KTO: 独立好/坏损失", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

# β 对 DPO 的影响
ax = axes[1, 1]
for b in [0.01, 0.05, 0.1, 0.5, 1.0]:
    y = -np.log(1 / (1 + np.exp(-b * x)) + 1e-10)
    ax.plot(x, y, lw=1.5, label=f"β={b}")
ax.set_title("β 对 DPO 损失的影响", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)

for ax in axes.flat:
    ax.set_xlabel("隐式奖励差")
    ax.set_ylabel("损失")

plt.suptitle("各对齐方法的损失函数对比", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. 总结与思考题

### 核心要点

1. **RLHF** 是对齐的奠基之作，但流程复杂、训练不稳定
2. **DPO** 大幅简化了流程，已成为中小模型的标配
3. **KTO/ORPO/SimPO** 在数据需求和内存上进一步优化
4. **GRPO** 是推理任务的新宠，被 DeepSeek-R1 验证
5. **CAI** 解决了人工标注的瓶颈，用 AI 替代人类
6. 选择方法要看：**数据类型、计算资源、任务类型**

### 思考题

1. **DPO 和 RLHF 在理论上等价，但实际效果有差异。** 你认为在什么场景下 RLHF 仍然优于 DPO？

2. **GRPO 适合有明确奖励函数的任务（如数学推理）。** 对于开放式对话任务，GRPO 的奖励函数应该如何设计？

3. **如果你要为一个 7B 参数的中文对话模型做对齐，你会选择哪种方法？** 说明你的理由和资源假设。

---
*恭喜完成 LLM Training Sprint 对齐部分的学习! 从 SFT 的局限到 RLHF，再到 DPO 和最新变体，你已经掌握了对齐技术的全貌。*